# Generate the table for displaying Pango X events

In [1]:

import numpy as np
import pandas as pd
import tszip
import sc2ts

In [2]:
ts = tszip.load("../data/sc2ts_viridian_v2.0.trees.tsz")
df_node = sc2ts.node_data(ts, inheritance_stats=False)

In [3]:
dfe = pd.read_csv("../data/pango_x_events.csv")
dfe

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,pango
0,1492869,XU,13,S,1,{},1252155,7.0,113.687852,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",XU
1,1822188,XCE,2,R,1,{},1822188,0.0,0.000000,{'XCE': 1},XCE
2,1837016,XCR,2,R,84,{'XBB.1.5.70': 1},1837016,0.0,0.000000,"{'XCR': 84, 'XBB.1.5.70': 1}",XCR
3,1844851,XCH,6,R,155,{},1844851,0.0,0.000000,"{'XCH': 103, 'XCH.1': 52}",XCH
4,1875327,XDD,0,R,112,{},1875327,0.0,0.000000,"{'XDD': 68, 'XDD.1.1': 17, 'XDD.1': 17, 'XDD.1...",XDD
...,...,...,...,...,...,...,...,...,...,...,...
93,1491345,XAC,2,I,9,{},1452574,2.0,71.352246,{'XAC': 19},XAC
94,1452573,XAC,2,S,1,{},1452574,2.0,78.842643,{'XAC': 19},XAC
95,1481871,XAC,1,S,1,{},1452574,2.0,98.842643,{'XAC': 19},XAC
96,1817163,XDB,7,S,1,{},1676682,11.0,330.161421,{'too_many_to_classify': 112056},XDB


In [4]:
conv1 = {}
conv2 = {}
descendants = {}
for k, row in dfe.iterrows():
    conv1[k] = eval(row["closest_recombinant_descendants"])
    conv2[k] = eval(row["non_pango_samples"])
    descendants[k] = {row["pango"]: row["pango_samples"], **conv2[k]}
dfe["closest_recombinant_descendants"] = conv1
dfe["non_pango_samples"] = conv2
dfe["descendants"] = descendants


In [5]:
dfr = pd.read_csv("../data/recombinants.csv").set_index("recombinant")
dfr

,sample_id,num_descendant_samples,num_samples,distinct_sample_pango,interval_left,interval_right,num_mutations,Viridian_amplicon_scheme,Artic_primer_version,date_added,...,parent_mrca_pango,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts
recombinant,,,,,,,,,,,,,,,,,,,,,
1441963,SRR19695004,1,1,1,519,670,2,COVID-AMPLISEQ-V1,.,2022-04-15,...,B.1.1.529,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3,2,55,False,6
1543488,ERR9939974,17,1,1,695,958,1,COVID-ARTIC-V4.1,.,2022-06-27,...,B.1.1.529,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,4,1,11,False,9
753471,SRR20259474,1,1,1,510,1222,1,COVID-AMPLISEQ-V1,.,2021-10-21,...,B.1.617.2,Delta (B.1.617.2-like),1273.399240,2020-12-11,False,2,2,19,False,5
1801171,ERR11184433,1,1,1,695,1545,2,COVID-ARTIC-V4.1,.,2023-03-20,...,B.1.1.529,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,12,1,42,False,10
1630771,ERR10150242,1,1,1,695,1598,1,COVID-ARTIC-V4.1,.,2022-08-21,...,B.1.1.529,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,3,0,13,False,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1234418,SRR18234565,16,2,1,25923,29734,2,COVID-VARSKIP-V1a-2b,.,2022-02-08,...,BA.1,Omicron (BA.1-like),953.290830,2021-10-27,False,2,6,3,False,5
1759455,ERR13041473,2,1,1,29509,29739,2,COVID-ARTIC-V3,.,2023-01-30,...,B.1.1.529,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,13,36,1,False,6
1558618,ERR12781803,6,1,1,29509,29739,1,COVID-ARTIC-V3,.,2022-07-06,...,B.1.1.529,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,6,24,1,False,5


In [6]:
dfr["mutations_averted"] = dfr["k1000_muts"] - dfr["num_mutations"]

In [7]:
dfe = dfe.join(dfr, "closest_recombinant")

## Number of pango X lineages in the ARG

There's 44 distinct pango X lineages (45 in the list, but we don't count XBB.1)

In [8]:
all_x_events = dfe["root_pango"].unique()
all_x_events

<StringArray>
[        'XU',        'XCE',        'XCR',        'XCH',        'XDD',
        'XAS',        'XBK',         'XH',        'XCF',        'XAL',
        'XAP',        'XBQ',        'XDY',      'XAY.2',        'XAZ',
        'XBR',        'XAG',        'XAV',        'XBE',         'XW',
        'XAU',         'XL',        'XAA',         'XE',        'XBS',
  'XBC.1.2.1',         'XB',        'XBV',        'XAM',        'XBM',
        'XCQ',        'XBF',        'XAB',        'XBG',         'XA',
        'XCP',         'XM',        'XBB',      'XBB.1',        'XCC',
    'XBB.1.5', 'XBB.1.5.16',    'XBB.2.3', 'XBB.1.16.4',        'XDP',
        'XDR',        'XCT',        'XCY',        'XBZ',        'XAD',
        'XAE',        'XBL',      'XDV.1',         'XF',         'XN',
         'XC',        'XDF',         'XV',        'XBH',        'XCL',
         'XP',         'XG',        'XDA',        'XDK',         'XQ',
        'XBW',        'XAN',        'XBD',         'XR',       

In [9]:
len(all_x_events)

77

In [10]:
xs = [x for x in df_node.pango.unique() if x.startswith("X") and "." not in x]
len(xs)

70

# List the events

List the events that are monophyletic for the pango lineage in question --- almost. XM is the main exception. 

We also exclude XAC and XAD, which are split and subsets of Xx+

In [11]:
counts = dfe["root_pango"].value_counts() 
multiple = counts[counts > 1]
multiple

root_pango
XBB.1.5    5
XBB.1      4
XDP        4
XAC        4
XM         3
XCE        2
XAS        2
XAP        2
XBV        2
XCC        2
XDB        2
Name: count, dtype: int64

In [13]:
dfe[dfe["root_pango"].isin(multiple.index)]

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts,mutations_averted
1,1822188,XCE,2,R,1,{},1822188,0.0,0.000000,{'XCE': 1},...,Omicron (BA.2-like),922.010023,2021-11-27,True,17.0,4.0,42.0,True,6.0,4.0
5,1626363,XAS,1,I,9,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1522026,XAS,1,S,75,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,1451097,XAP,1,I,17,{},1131641,3.0,51.134801,"{'BA.5.2.1': 39577, 'BA.5.1': 28469, 'BA.5.2':...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,4.0,43.0,6.0,True,7.0,7.0
12,1459103,XAP,1,S,4,{},1131641,4.0,98.000000,"{'BA.5.2.1': 39577, 'BA.5.1': 28469, 'BA.5.2':...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,4.0,43.0,6.0,True,7.0,7.0
29,1722702,XBV,3,R,3,{},1722702,0.0,0.000000,"{'XBV': 3, 'XBB.1': 1}",...,Omicron (BA.2-like),922.010023,2021-11-27,True,11.0,14.0,24.0,True,17.0,14.0
38,1095890,XM,0,R,1,{'BA.1': 1},1095890,0.0,0.000000,"{'XM': 1, 'BA.1': 1}",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3.0,30.0,15.0,True,30.0,30.0
39,1182466,XM,0,R,31,"{'BA.2': 7, 'XAL': 14}",1182466,0.0,0.000000,"{'XM': 31, 'XAL': 14, 'BA.2': 7}",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3.0,27.0,23.0,True,9.0,8.0
40,1360863,XM,1,R,1,{},1360863,0.0,0.000000,{'XM': 1},...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3.0,29.0,21.0,True,5.0,4.0
42,1743137,XBB.1,0,R,1,{},1743137,0.0,0.000000,{'XBB.1': 1},...,Omicron (BA.2-like),922.010023,2021-11-27,True,10.0,6.0,28.0,True,6.0,6.0


In [14]:
dfe[dfe["root_pango"].isin(multiple.index)]["pango_samples"].sum()

np.int64(497)

Aside from the major XM node, accounting for 26 samples, there's 23 we're not counting here.


These are messy - let's leave everything except the dominant XM out and see where we go. XAC and XAD are all under the same recombinaiont, which we'll probably analyse separately.

In [18]:
pango_events_tbl = dfe[~dfe["root_pango"].isin(["XM", "XAC", "XAD"])]
pango_events_tbl = pd.concat([pango_events_tbl, dfe[dfe["root"] == 1003220]]).copy()
pango_events_tbl

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts,mutations_averted
0,1492869,XU,13,S,1,{},1252155,7.0,113.687852,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,7.0,44.0,True,5.0,4.0
1,1822188,XCE,2,R,1,{},1822188,0.0,0.000000,{'XCE': 1},...,Omicron (BA.2-like),922.010023,2021-11-27,True,17.0,4.0,42.0,True,6.0,4.0
2,1837016,XCR,2,R,84,{'XBB.1.5.70': 1},1837016,0.0,0.000000,"{'XCR': 84, 'XBB.1.5.70': 1}",...,Omicron (XBB.1-like),639.779580,2022-09-06,True,8.0,18.0,11.0,True,12.0,10.0
3,1844851,XCH,6,R,155,{},1844851,0.0,0.000000,"{'XCH': 103, 'XCH.1': 52}",...,Omicron (XBB.1-like),639.779580,2022-09-06,True,6.0,18.0,5.0,True,11.0,3.0
4,1875327,XDD,0,R,112,{},1875327,0.0,0.000000,"{'XDD': 68, 'XDD.1.1': 17, 'XDD.1': 17, 'XDD.1...",...,Omicron (BA.2-like),922.010023,2021-11-27,True,14.0,62.0,11.0,True,67.0,67.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,1538224,XAJ,11,I,21,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,1173316,XS,0,R,19,{},1173316,0.0,0.000000,{'XS': 19},...,.,1591.515332,2020-01-28,True,7.0,14.0,76.0,True,13.0,13.0
91,1383518,XZ,1,I,51,{},1131641,4.0,52.000000,"{'BA.5.2.1': 39577, 'BA.5.1': 28469, 'BA.5.2':...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,4.0,43.0,6.0,True,7.0,7.0
96,1817163,XDB,7,S,1,{},1676682,11.0,330.161421,{'too_many_to_classify': 112056},...,Omicron (BA.2-like),922.010023,2021-11-27,True,6.0,13.0,10.0,True,21.0,7.0


Also remove XBB.1. Even though it's a solid recomb, it's not the root of all XBB.1s, and so it's complicating the story. We should mention in the text or somewhere, though.

In [19]:
pango_events_tbl = pango_events_tbl[~(pango_events_tbl["root_pango"] == "XBB.1")].copy()
pango_events_tbl.shape[0]

86

In [20]:

def format_parent_col(df_in, parent_col, pango_col):
    parent_pango = {}
    for k, row in df_in.iterrows():
        
        focal = int(row["root"])
        parent = row[parent_col]
        if parent == -1 or np.isnan(parent):
            continue
        parent = int(parent)
        node_row = df_node.loc[parent]
        pango = node_row["pango"]
        #print(row)
        #print(node_row)
        if node_row["is_sample"]:
            is_sample = True
        else:
            children = list(ts.edges_child[ts.edges_parent == parent])
            if focal in children:
                children.remove(focal)
            df = df_node.loc[children]
            is_sample = np.any(df["is_sample"] & (df["num_mutations"] == 0))
        if is_sample:
            parent_pango[k] = "\\textbf{" + pango + "}"
        else:
            parent_pango[k] = pango
    df_in[pango_col] = parent_pango

In [21]:
format_parent_col(pango_events_tbl, "parent_left", "parent_left_pango")
format_parent_col(pango_events_tbl, "parent_right", "parent_right_pango")

# Examine nested BA.2 samples

In [22]:
df_nested = pango_events_tbl[pango_events_tbl["non_pango_samples"] != {}]
df_nested

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts,mutations_averted
2,1837016,XCR,2,R,84,{'XBB.1.5.70': 1},1837016,0.0,0.000000,"{'XCR': 84, 'XBB.1.5.70': 1}",...,Omicron (XBB.1-like),639.779580,2022-09-06,True,8.0,18.0,11.0,True,12.0,10.0
7,1728287,XBK,1,I,178,{'CJ.1': 1},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,1378600,XW,1,R,37,{'BA.2': 3},1378600,0.0,0.000000,"{'XW': 37, 'BA.2': 3}",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,4.0,4.0,44.0,True,5.0,4.0
41,1676682,XBB,15,R,112040,"{'BQ.1.1.13': 2, 'JN.1.1': 11, 'JN.3': 1, 'JN....",1676682,0.0,0.000000,{'too_many_to_classify': 112056},...,Omicron (BA.2-like),922.010023,2021-11-27,True,6.0,13.0,10.0,True,21.0,7.0
57,1871948,XDP,0,R,105,{'JN.1.4': 3},1871948,0.0,0.000000,"{'XDP': 62, 'XDP.1': 24, 'XDP.3': 13, 'XEB': 5...",...,Omicron (BA.2-like),922.010023,2021-11-27,True,12.0,71.0,6.0,True,7.0,6.0
58,1878215,XDP,0,R,32,{'JN.1.4': 2},1878215,0.0,0.000000,"{'XDP': 18, 'XDP.1': 8, 'XEB': 4, 'JN.1.4': 2,...",...,Omicron (BA.2-like),922.010023,2021-11-27,True,11.0,67.0,6.0,True,5.0,5.0
70,1871948,XDP,0,R,100,"{'XEB': 5, 'JN.1.4': 3}",1871948,0.0,0.000000,"{'XDP': 62, 'XDP.1': 24, 'XDP.3': 13, 'XEB': 5...",...,Omicron (BA.2-like),922.010023,2021-11-27,True,12.0,71.0,6.0,True,7.0,6.0
71,1878215,XDP,0,R,28,"{'JN.1.4': 2, 'XEB': 4}",1878215,0.0,0.000000,"{'XDP': 18, 'XDP.1': 8, 'XEB': 4, 'JN.1.4': 2,...",...,Omicron (BA.2-like),922.010023,2021-11-27,True,11.0,67.0,6.0,True,5.0,5.0
81,1252155,XQ,1,R,58,"{'BA.2': 42, 'XR': 18, 'XAA': 22, 'XU': 1, 'XA...",1252155,0.0,0.000000,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,7.0,44.0,True,5.0,4.0
85,1364349,XR,4,I,18,{'BA.2': 1},1252155,4.0,21.687852,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,7.0,44.0,True,5.0,4.0


In [23]:
# All the nested samples are either BA.2 or from X lineages
for d in df_nested["non_pango_samples"].values:
    for k in d:
        assert k.startswith("X") or k == "BA.2"

AssertionError: 

In [24]:
# Get the amiguous nodes from the tree sequence
descendants = {row["root"]: set() for _, row in df_nested.iterrows()}
for tree in ts.trees():
    for root, desc in descendants.items():
        for u in tree.samples(root):
            if ts.node(u).metadata["pango"] == "BA.2":
                desc.add(u)
descendants    

KeyboardInterrupt: 

In [ ]:
df_pangolin = pd.read_csv("../arg_postprocessing/sc2ts_v1_2023-02-21_pr_pp_mp_aph_bps.lineage_report.csv").set_index("taxon")
df_pangolin

In [23]:
data = []
pango_lookup = pango_events_tbl.set_index("root")
for root, desc in descendants.items():
    for u in desc:
        row = df_pangolin.loc[f"n{u}"]
        data.append({"root": root, "node": u, 
                     "pango":pango_lookup.loc[root]["pango"],
                     **dict(row)})
df_ba2_calls = pd.DataFrame(data)


## XJ

The three nested BA.2 calls under XJ could equally equally have been called XJ by the Usher placements.

In [24]:
df_ba2_calls[df_ba2_calls["pango"] == "XJ"][["scorpio_call","conflict", "note"]]

,scorpio_call,conflict,note
0,Omicron (Unassigned),0.5,Usher placements: BA.2(1/2) XJ(1/2); scorpio f...
1,Omicron (Unassigned),0.5,Usher placements: BA.2(1/2) XJ(1/2); scorpio f...
2,Omicron (Unassigned),0.5,Usher placements: BA.2(1/2) XJ(1/2); scorpio f...


## XR

The single nested XR sample was unambiguously assinged by Usher

In [25]:
df_ba2_calls[df_ba2_calls["pango"] == "XR"][["scorpio_call", "conflict", "note"]]

,scorpio_call,conflict,note
3,Omicron (BA.2-like),0.0,Usher placements: BA.2(1/1)


## XQ

2 of the 37 embedded samples could have been designated XU, otherwise there are all confident BA.2 placments (some done from lineage hashes also)

In [26]:
df_ba2_calls[df_ba2_calls["pango"] == "XQ"][["scorpio_call", "conflict", "note"]].reset_index()

,index,scorpio_call,conflict,note
0,4,Omicron (BA.2-like),0.0,Usher placements: BA.2(3/3)
1,5,Omicron (Unassigned),0.0,Usher placements: BA.2(3/3); scorpio found ins...
2,6,Omicron (BA.2-like),0.0,Usher placements: BA.2(1/1)
3,7,Omicron (BA.2-like),0.0,Usher placements: BA.2(1/1)
4,8,Omicron (Unassigned),0.0,Usher placements: BA.2(3/3); scorpio found ins...
5,9,Omicron (Unassigned),0.0,Usher placements: BA.2(3/3); scorpio found ins...
6,10,Omicron (BA.2-like),0.5,Usher placements: BA.2(1/2) XU(1/2)
7,11,Omicron (BA.2-like),0.5,Usher placements: BA.2(1/2) XU(1/2)
8,12,Omicron (BA.2-like),0.0,Usher placements: BA.2(1/1)
9,13,Omicron (BA.2-like),0.0,Usher placements: BA.2(1/1)


## XN

The one nested pango here could have been called as XN (2 of the 3 placements were XN?)

In [27]:
df_ba2_calls[df_ba2_calls["pango"] == "XN"][["conflict", "note"]].reset_index()

,index,conflict,note
0,41,0.666667,Usher placements: BA.2(1/3) XN(2/3)


## XBB

The single XBB sample seems to have a strong pango and Scorpio call for BA.2

In [28]:
df_ba2_calls[df_ba2_calls["pango"] == "XBB"][["scorpio_call", "conflict", "note"]].reset_index()

,index,scorpio_call,conflict,note
0,42,Omicron (BA.2-like),0.0,Usher placements: BA.2(1/1)


## XM

All got BA.2 assigments, with scorpio finding insufficient support and giving generic "Omicron (Unassigned)"

In [29]:
df_ba2_calls[df_ba2_calls["pango"] == "XM"][["scorpio_call", "conflict", "note"]].reset_index()

,index,scorpio_call,conflict,note
0,43,Omicron (Unassigned),0.0,Usher placements: BA.2(2/2); scorpio found ins...
1,44,Omicron (Unassigned),0.0,Usher placements: BA.2(2/2); scorpio found ins...
2,45,Omicron (Unassigned),0.0,Usher placements: BA.2(3/3); scorpio found ins...
3,46,Omicron (Unassigned),0.0,Usher placements: BA.2(1/1); scorpio found ins...
4,47,Omicron (Unassigned),0.0,Usher placements: BA.2(3/3); scorpio found ins...
5,48,Omicron (Unassigned),0.0,Usher placements: BA.2(3/3); scorpio found ins...
6,49,Omicron (Unassigned),0.0,Usher placements: BA.2(2/2); scorpio found ins...
7,50,Omicron (Unassigned),0.0,Usher placements: BA.2(2/2); scorpio found ins...
8,51,Omicron (Unassigned),0.0,Usher placements: BA.2(1/1); scorpio found ins...
9,52,Omicron (Unassigned),0.0,Usher placements: BA.2(2/2); scorpio found ins...


In [30]:
df_ba2_calls[df_ba2_calls["pango"] == "XM"]["note"].values[0]

'Usher placements: BA.2(2/2); scorpio found insufficient support to assign a specific lineage'

## Overall

Overall, we've got 59 samples with BA.2 assignments. There are highly enriched for ambiguous calls, with 6/59 having multiple potential placements by Usher (10%) versus 2.75% over 2,747,985 nodes, and 25 of the 357 "Omicron (Unassigned)" Scorpio calls.



In [25]:
def format(n):
    return f"{n * 100:4g}%"

In [32]:
df_ba2_calls.shape[0], df_pangolin.shape[0]

(59, 2747985)

In [33]:
np.sum(df_ba2_calls["conflict"] > 0)

np.int64(6)

In [34]:
format(np.sum(df_ba2_calls["conflict"] > 0) / df_ba2_calls.shape[0])

'10.1695%'

In [35]:
format(np.sum(df_pangolin["conflict"] > 0) / df_pangolin.shape[0])

'2.7486%'

In [36]:
np.sum(df_ba2_calls["scorpio_call"] == "Omicron (Unassigned)")

np.int64(25)

In [37]:
np.sum(df_pangolin["scorpio_call"] == "Omicron (Unassigned)")

np.int64(357)

In [38]:
format(np.sum(df_ba2_calls["scorpio_call"] == "Omicron (Unassigned)") /
      np.sum(df_pangolin["scorpio_call"] == "Omicron (Unassigned)"))

'7.0028%'



# Type I RE events: monophyletic for single pango

In [26]:
type_1_events = pango_events_tbl[(pango_events_tbl["root_type"] == "R")]
type_1_events = type_1_events.sort_values(["mutations_averted", "root_pango"], ascending=False)
type_1_events

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts,mutations_averted
4,1875327,XDD,0,R,112,{},1875327,0.0,0.0,"{'XDD': 68, 'XDD.1.1': 17, 'XDD.1': 17, 'XDD.1...",...,Omicron (BA.2-like),922.010023,2021-11-27,True,14.0,62.0,11.0,True,67.0,67.0
60,1843410,XCT,3,R,57,{},1843410,0.0,0.0,"{'XCT.1': 32, 'XCT': 25}",...,Omicron (BA.2-like),922.010023,2021-11-27,True,19.0,27.0,34.0,True,29.0,26.0
17,1732233,XBR,1,R,3,{},1732233,0.0,0.0,{'XBR': 3},...,Omicron (BA.2-like),922.010023,2021-11-27,True,13.0,25.0,25.0,True,26.0,24.0
26,1731957,XBS,3,R,19,{},1731957,0.0,0.0,{'XBS': 19},...,Omicron (BA.2-like),922.010023,2021-11-27,True,12.0,21.0,20.0,True,23.0,20.0
36,132899,XA,0,R,39,{},132899,0.0,0.0,{'XA': 39},...,.,1591.515332,2020-01-28,True,4.0,20.0,26.0,True,19.0,19.0
66,1894585,XDV.1,8,R,32,{},1894585,0.0,0.0,{'XDV.1': 32},...,Omicron (BA.2-like),922.010023,2021-11-27,True,9.0,27.0,43.0,True,27.0,17.0
52,1799712,XBB.2.3,1,R,1,{},1799712,0.0,0.0,{'XBB.2.3': 1},...,Omicron (BA.2-like),922.010023,2021-11-27,True,15.0,17.0,27.0,True,18.0,17.0
29,1722702,XBV,3,R,3,{},1722702,0.0,0.0,"{'XBV': 3, 'XBB.1': 1}",...,Omicron (BA.2-like),922.010023,2021-11-27,True,11.0,14.0,24.0,True,17.0,14.0
44,1722702,XBV,3,R,3,{},1722702,0.0,0.0,"{'XBV': 3, 'XBB.1': 1}",...,Omicron (BA.2-like),922.010023,2021-11-27,True,11.0,14.0,24.0,True,17.0,14.0
90,1173316,XS,0,R,19,{},1173316,0.0,0.0,{'XS': 19},...,.,1591.515332,2020-01-28,True,7.0,14.0,76.0,True,13.0,13.0


In [27]:
col_name_map = {
    "root_pango": "pango",
    "mutations_averted": "averted",
    "parent_left_pango": "left p",
    "parent_right_pango": "right p",
    "interval_left": "left",
    "interval_right": "right",
    "descendants": "descendants",
    #"pango_samples": "samples", 
    #"non_pango_samples": "nested",
}
s = type_1_events[list(col_name_map.keys())].to_latex(
        escape=False, index=False, header=list(col_name_map.values()),
        float_format="%.0f")
s = s.replace("'", "").replace(": ", ":")
#s = s.replace("{BA.2:37, XR:17, XAA:17, XU:1, XAM:21, XAG:6\}", "{BA.2:37, $\star$\}")
print(s)

\begin{tabular}{lrllrrl}
\toprule
pango & averted & left p & right p & left & right & descendants \\
\midrule
XDD & 67 & JN.1.1 & EG.5.1.1 & 25700 & 26275 & {XDD:112} \\
XCT & 26 & \textbf{JG.4} & DV.7.1 & 18584 & 19326 & {XCT:57} \\
XBR & 24 & \textbf{BN.3.1} & \textbf{BQ.1.25} & 22034 & 22190 & {XBR:3} \\
XBS & 20 & BY.1 & \textbf{BQ.1.1} & 22034 & 22190 & {XBS:19} \\
XA & 19 & \textbf{B.1.177.18} & \textbf{B.1.1.7} & 21256 & 21765 & {XA:39} \\
XDV.1 & 17 & XBB.1.19 & JN.1 & 19327 & 21363 & {XDV:32} \\
XBB.2.3 & 17 & \textbf{BQ.1.1} & \textbf{XBB.2.3} & 17860 & 19326 & {XBB:1} \\
XBV & 14 & CR.1.3 & \textbf{XBB.1} & 19327 & 21765 & {XBV:3} \\
XBV & 14 & CR.1.3 & \textbf{XBB.1} & 19327 & 21765 & {XBB:3} \\
XS & 13 & AY.103 & \textbf{BA.1.1} & 9054 & 10449 & {XS:19} \\
XCZ & 13 & \textbf{EG.5.1.1} & GK.1.1 & 18493 & 21718 & {XCZ:6} \\
XCY & 10 & \textbf{EG.5.1.3} & GK.3.1 & 21719 & 22480 & {XCY:1} \\
XCR & 10 & GK.1.1 & FU.1.1 & 23674 & 25045 & {XCR:84, XBB.1.5.70:1} \\
XCP & 9 & \text

## Type II RE events. 


In [28]:
ba5_root = 1189192
type_3_events = pango_events_tbl[
    (pango_events_tbl["root_type"] != "R") & ~(pango_events_tbl["closest_recombinant"].isin([-1, ba5_root]))].copy()
type_3_events

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts,mutations_averted
0,1492869,XU,13,S,1,{},1252155,7.0,1.136879e+02,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,7.0,44.0,True,5.0,4.0
8,1070425,XH,3,S,40,{},1070426,1.0,8.000000e+00,"{'XE': 1256, 'XH': 40, 'BA.2': 34}",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3.0,14.0,37.0,True,16.0,13.0
9,1825239,XCF,5,I,45,{},1770940,4.0,2.392404e+01,"{'XCF': 38, 'XBB.1.5': 6, 'XCF.3': 6, 'XBB.1.5...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,25.0,1.0,False,5.0,4.0
10,1505519,XAL,2,I,14,{},1182466,4.0,7.131609e+01,"{'XM': 31, 'XAL': 14, 'BA.2': 7}",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3.0,27.0,23.0,True,9.0,8.0
11,1451097,XAP,1,I,17,{},1131641,3.0,5.113480e+01,"{'BA.5.2.1': 39577, 'BA.5.1': 28469, 'BA.5.2':...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,4.0,43.0,6.0,True,7.0,7.0
12,1459103,XAP,1,S,4,{},1131641,4.0,9.800000e+01,"{'BA.5.2.1': 39577, 'BA.5.1': 28469, 'BA.5.2':...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,4.0,43.0,6.0,True,7.0,7.0
18,1489804,XAG,4,I,17,{},1252155,7.0,4.824392e+01,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,7.0,44.0,True,5.0,4.0
19,1528535,XAV,3,S,13,{},1412770,7.0,8.500932e+01,{'too_many_to_classify': 244566},...,Probable Omicron (Unassigned),1492.502163,2020-05-06,False,2.0,7.0,5.0,True,10.0,7.0
24,1410182,XAA,3,I,22,{},1252155,7.0,4.368785e+01,"{'XQ': 58, 'BA.2': 42, 'XAM': 26, 'XAA': 22, '...",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,5.0,7.0,44.0,True,5.0,4.0
25,1132512,XE,4,I,1256,{},1070426,2.0,9.574537e+00,"{'XE': 1256, 'XH': 40, 'BA.2': 34}",...,Probable Omicron (Unassigned),1492.502163,2020-05-06,True,3.0,14.0,37.0,True,16.0,13.0


In [29]:
set(type_3_events["closest_recombinant"]) - set(type_1_events["root"])

{242115, 463337, 1070426, 1092999, 1131641, 1182466, 1412770, 1770940}

In [30]:
#direct_events = {row["root"]: row["root_pango"] for _, row in type_1_events.iterrows()}

event_aliases = {
    #-1: "non_recomb",
    #1189192: "non_recomb (BA.5)",
    965353: "XE/XH",
    966905: "XJ",
    964555: "Xx",
    1177107: "XAF", # Weird one  
    1348822: "XBM",
    }

event_aliases

{965353: 'XE/XH', 966905: 'XJ', 964555: 'Xx', 1177107: 'XAF', 1348822: 'XBM'}

In [44]:
assert set(event_aliases.keys()) == set(type_3_events["closest_recombinant"]) - set(type_1_events["root"])

In [31]:
type_2_events = pango_events_tbl[pango_events_tbl["closest_recombinant"].isin(event_aliases.keys())]
type_2_events = type_2_events.groupby("closest_recombinant").first().sort_values("mutations_averted", ascending=False)
type_2_events["alias"] = event_aliases
type_2_events[["root_pango", "alias", "closest_recombinant_descendants", "mutations_averted"]]

,root_pango,alias,closest_recombinant_descendants,mutations_averted
closest_recombinant,,,,
965353,NaN,XE/XH,NaN,NaN
966905,NaN,XJ,NaN,NaN
964555,NaN,Xx,NaN,NaN
1177107,NaN,XAF,NaN,NaN
1348822,NaN,XBM,NaN,NaN


In [32]:
col_name_map = {
    "alias": "alias",
    "mutations_averted": "averted",
    "parent_left_pango": "left p",
    "parent_right_pango": "right p",
    "interval_left": "left",
    "interval_right": "right",
    "closest_recombinant_descendants": "descendants",
}
s = type_2_events[list(col_name_map.keys())].to_latex(
        escape=False, index=False, header=list(col_name_map.values()),
        float_format="%.0f")
s = s.replace("'", "").replace(": ", ":")
#s = s.replace("{BA.2:37, XR:17, XAA:17, XU:1, XAM:21, XAG:6\}", "{BA.2:37, $\star$\}")
print(s)

\begin{tabular}{lrllrrl}
\toprule
alias & averted & left p & right p & left & right & descendants \\
\midrule
XE/XH & NaN & NaN & NaN & NaN & NaN & NaN \\
XJ & NaN & NaN & NaN & NaN & NaN & NaN \\
Xx & NaN & NaN & NaN & NaN & NaN & NaN \\
XAF & NaN & NaN & NaN & NaN & NaN & NaN \\
XBM & NaN & NaN & NaN & NaN & NaN & NaN \\
\bottomrule
\end{tabular}



In [33]:
event_names = {**event_aliases, **{row["root"]: row["root_pango"] for _, row in type_1_events.iterrows()}}

event_names

{965353: 'XE/XH',
 966905: 'XJ',
 964555: 'Xx',
 1177107: 'XAF',
 1348822: 'XBM',
 1875327: 'XDD',
 1843410: 'XCT',
 1732233: 'XBR',
 1731957: 'XBS',
 132899: 'XA',
 1894585: 'XDV.1',
 1799712: 'XBB.2.3',
 1722702: 'XBV',
 1173316: 'XS',
 1837724: 'XCZ',
 1851951: 'XCY',
 1837016: 'XCR',
 1836124: 'XCP',
 1769335: 'XCC',
 1558028: 'XBG',
 1839568: 'XCL',
 1633571: 'XBM',
 1670327: 'XBD',
 1708787: 'XBF',
 1676682: 'XBB',
 1134481: 'XJ',
 1871948: 'XDP',
 1874829: 'XDK',
 1791222: 'XBW',
 1671867: 'XBH',
 1107501: 'XF',
 1886690: 'XDR',
 1878215: 'XDP',
 1791503: 'XBB.1.5',
 1502510: 'XAN',
 1326725: 'XAE',
 1417358: 'XY',
 1378600: 'XW',
 1301504: 'XV',
 1252155: 'XQ',
 1222133: 'XL',
 1895676: 'XDY',
 1829166: 'XCQ',
 1822188: 'XCE',
 1627107: 'XBE',
 1787625: 'XBB.1.5.16',
 1767951: 'XBB.1.5',
 1755170: 'XBB.1.5',
 1786345: 'XBB.1.5',
 1796262: 'XBB.1.5',
 1832383: 'XBB.1.16.4',
 1488660: 'XAZ',
 1866449: 'XDA',
 1844851: 'XCH',
 1730254: 'XBC.1.2.1',
 1726834: 'XAY.2'}

In [34]:
type_3_events["event_name"] = [event_names[u] for u in type_3_events["closest_recombinant"]]
type_3_events = type_3_events.sort_values(["event_name", "pango"])
type_3_events

KeyError: 1070426

In [49]:
col_name_map = {
    "root_pango": "pango",
    "event_name": "name",
    "closest_recombinant_path_len": "path len",
    "closest_recombinant_time": "time",
    "root_mutations": "mutations",
    "descendants": "descendants",
    #"non_pango_samples": "nested",
  
    #"closest_recombinant_descendants": "by pango",
}
s = type_3_events[list(col_name_map.keys())].to_latex(
        escape=True, index=False, header=list(col_name_map.values()),
        float_format="%.0f")
s = s.replace("'", "").replace(": ", ":")
print(s)

\begin{tabular}{llrrrl}
\toprule
pango & name & path len & time & mutations & descendants \\
\midrule
XAF & XAF & 5 & 140 & 8 & {XAF:1} \\
XBM & XBM & 1 & 23 & 1 & {XBM:10} \\
XE & XE/XH & 1 & 19 & 2 & {XE:1116} \\
XH & XE/XH & 1 & 55 & 7 & {XH:2} \\
XJ & XJ & 1 & 5 & 2 & {XJ:68, BA.2:3} \\
XAL & XM & 3 & 65 & 2 & {XAL:3} \\
XAA & XQ & 6 & 33 & 3 & {XAA:17} \\
XAG & XQ & 6 & 53 & 4 & {XAG:6} \\
XAM & XQ & 6 & 40 & 3 & {XAM:21} \\
XR & XQ & 3 & 11 & 4 & {XR:17, BA.2:1} \\
XU & XQ & 6 & 103 & 13 & {XU:1} \\
XAE & Xx & 2 & 47 & 7 & {XAE:9} \\
XAP & Xx & 4 & 66 & 1 & {XAP:20} \\
XZ & Xx & 4 & 48 & 1 & {XZ:48} \\
\bottomrule
\end{tabular}



# Not recombinants


In [35]:
type_4_events = pango_events_tbl[(pango_events_tbl["closest_recombinant"].isin([-1, ba5_root]))].copy()
type_4_events = type_4_events.sort_values(["root_mutations", "root_pango"], ascending=True)
type_4_events

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,...,parent_mrca_scorpio,parent_mrca_time,parent_mrca_date,is_rebar_recombinant,parent_pangonet_distance,net_min_supporting_loci_lft,net_min_supporting_loci_rgt,net_min_supporting_loci_lft_rgt_ge_4,k1000_muts,mutations_averted
5,1626363,XAS,1,I,9,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1522026,XAS,1,S,75,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,1476400,XAU,1,S,9,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1728287,XBK,1,I,178,{'CJ.1': 1},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,1736919,XBQ,1,I,36,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
68,1257906,XN,1,S,141,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,1295295,XP,1,S,48,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,1538224,XAJ,11,I,21,{},-1,inf,inf,{},...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
tree = ts.first()
root_parent = {}
root_parent_pango = {}
for k, row in type_4_events.iterrows():  
    u = tree.parent_array[row["root"]]
    root_parent[k] = u
    root_parent_pango[k] = df_node.loc[u, "pango"]

type_4_events["root_parent"] = root_parent
type_4_events["root_parent_pango"] = root_parent_pango


In [58]:
format_parent_col(type_4_events, "root_parent", "root_parent_pango")


In [37]:
col_name_map = {
    "root_pango": "pango",
    "root_mutations": "mutations",
    "root_parent_pango": "parent",
    #"parent_is_observed_sample": "parent sample",
    "descendants": "descandants",
  
    #"closest_recombinant_descendants": "by pango",
}
s = type_4_events[list(col_name_map.keys())].to_latex(
        escape=False, index=False, header=list(col_name_map.values()),
        float_format="%.0f")
s = s.replace("'", "").replace(": ", ":")
print(s)

\begin{tabular}{lrll}
\toprule
pango & mutations & parent & descandants \\
\midrule
XAS & 1 & BA.4 & {XAS:9} \\
XAS & 1 & BA.4 & {XAS:75} \\
XAU & 1 & BA.2 & {XAU:9} \\
XBK & 1 & CJ.1 & {XBK:178, CJ.1:1} \\
XBQ & 1 & CJ.1 & {XBQ:36} \\
XN & 1 & BA.2 & {XN:141} \\
XP & 1 & BA.1.1 & {XP:48} \\
XAJ & 11 & BA.2.12 & {XAJ:21} \\
\bottomrule
\end{tabular}

